# Module 0.2: Neural Networks in 30 Minutes

**Goal:** build the mental model you'll use for the rest of the course. By the end you'll know what a neuron, a parameter, a loss, and gradient descent *actually are* — and you'll have trained a tiny network with your own hands. Everything later (attention, transformers, LLMs) is just this idea, scaled up.

You just met PyTorch tensors and `autograd` in **Module 0.1**. Now we connect those tools to the big picture.

In [ ]:
import torch
import torch.nn as nn
torch.manual_seed(0)  # reproducibility: same random numbers every run

## 1. What is a neuron?

### The "weighing evidence" analogy
Imagine you're deciding whether to go for a run. You weigh a few pieces of evidence:

| Evidence (input) | How much it matters (weight) |
| :--- | :--- |
| Is it sunny? | +2.0 (helps a lot) |
| Am I tired? | -3.0 (big turn-off) |
| Do I have free time? | +1.0 (helps a little) |

A **neuron** does exactly this. It takes some inputs, multiplies each by a **weight** (how much that input matters), adds them all up, adds a **bias** (a baseline nudge, like "I'm generally lazy: -1"), and then runs the result through an **activation function** (a final yes/no-ish dial).

$$\text{output} = \text{activation}\Big( w_1 x_1 + w_2 x_2 + \dots + b \Big)$$

That's the entire idea. A neuron is a weighted vote followed by a squashing dial.

In [ ]:
# One neuron, by hand.
inputs  = torch.tensor([1.0, 1.0, 0.0])  # sunny=yes, tired=yes, free time=no
weights = torch.tensor([2.0, -3.0, 1.0]) # how much each input matters
bias    = torch.tensor(-1.0)             # baseline laziness

weighted_sum = torch.dot(inputs, weights) + bias   # 2*1 + (-3)*1 + 1*0 - 1 = -2
activated    = torch.relu(weighted_sum)            # ReLU: keep positives, clamp negatives to 0

print(f"Weighted sum (before activation): {weighted_sum.item():.1f}")
print(f"After ReLU activation:  {activated.item():.1f}  (negative -> 0, so: don't run)")

### Why an activation function?
Without it, a neuron is just a straight line (a linear function). Stacking straight lines only ever gives you... another straight line. The activation introduces a **bend** (non-linearity), and bending is what lets networks model curves, corners, and complicated patterns. **ReLU** is the most common one — it's brutally simple: `max(0, x)`. It keeps positive signals and zeroes out negative ones.

## 2. What is a "parameter"?

The **weights** and **biases** are the neuron's *learnable numbers* — collectively, its **parameters**. They start out random (the network knows nothing) and training slowly tweaks them until the network is good at its job.

When you hear **"a 7B-parameter model"**, it means the network has 7 *billion* of these weight-and-bias numbers, all nudged into place by training. That's it. "Parameters" is not jargon for something mysterious — it's just the count of adjustable dials inside the network.

## 3. A layer = many neurons. A network = stacked layers.

One neuron is weak. But put many neurons side by side, each with its own weights, and you get a **layer** — they all look at the same inputs but learn to detect different things.

In PyTorch, a full layer of neurons is just **`nn.Linear(in_features, out_features)`**. It bundles all the weights and biases for you. `nn.Linear(3, 4)` means *"take 3 inputs, produce 4 outputs"* — i.e. 4 neurons, each looking at the same 3 inputs.

Stack a few of these, with an activation (ReLU) between them to add the bends, and you have a **neural network**. `nn.Sequential` just chains the layers in order — data flows top to bottom.

In [ ]:
# A tiny network: 1 input -> hidden layer of 16 neurons -> 1 output.
model = nn.Sequential(
    nn.Linear(1, 16),  # layer 1: 16 neurons, each sees the 1 input
    nn.ReLU(),         # the bend (non-linearity)
    nn.Linear(16, 16), # layer 2: 16 more neurons
    nn.ReLU(),         # another bend
    nn.Linear(16, 1),  # output layer: 1 number out
)

# Count the parameters (all the weights + biases combined).
total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal learnable parameters: {total_params}")
print("A real LLM has ~7,000,000,000 of these. Same idea, more dials.")

## 4. The forward pass: data in -> prediction out

Running data *through* the network (input flows forward, layer by layer, to produce an output) is called the **forward pass**. In PyTorch you just call the model like a function: `model(x)`.

Watch the **shapes**. We feed in a batch of 5 examples, each with 1 feature, and get back 5 predictions.

In [ ]:
x = torch.randn(5, 1)        # 5 examples, 1 feature each  -> shape (5, 1)
predictions = model(x)       # forward pass

print(f"Input shape:      {tuple(x.shape)}   (5 examples, 1 feature)")
print(f"Output shape:     {tuple(predictions.shape)}   (5 examples, 1 prediction each)")
print(f"\nPredictions (random for now -- the network is untrained):\n{predictions.squeeze().tolist()}")

The predictions are garbage right now — the weights are random, so the network is guessing. We need a way to *measure* how bad the guesses are, and then a way to *fix* them. Those are the next two ideas.

## 5. Loss: a single number measuring "how wrong"

To improve, the network needs a score for how wrong it is. That score is the **loss**. Lower is better; **0 means perfect**.

For predicting a number (a *regression* task, like our toy below), the classic loss is **Mean Squared Error (MSE)**: take each prediction, subtract the true answer, square the difference (so over- and under-shooting both count as positive error), and average over all examples.

$$\text{MSE} = \frac{1}{N}\sum_{i=1}^{N} (\text{prediction}_i - \text{target}_i)^2$$

*(For classification — "is this a cat or a dog?" — a different loss called cross-entropy is used. You'll meet it in Module 5.1. The idea is the same: one number for how wrong.)*

In [ ]:
loss_fn = nn.MSELoss()

fake_targets = torch.zeros(5, 1)             # pretend the right answer is 0 for all 5
loss = loss_fn(predictions, fake_targets)    # how far off are we, on average (squared)?
print(f"Loss (how wrong the untrained network is): {loss.item():.4f}")
print("Training's whole job: push this number DOWN toward 0.")

## 6. Gradient descent: which way do we nudge each dial?

We have a number (the loss) we want to shrink, and thousands of dials (parameters) we can turn. **Which way should we turn each one?**

This is where **autograd** (from Module 0.1) earns its keep. When you call `loss.backward()`, PyTorch computes the **gradient** for every parameter — a number that says: *"if you increase this weight a tiny bit, the loss goes up by this much (or down, if negative)."* The gradient is the slope of the loss with respect to that weight.

**Gradient descent** is then dead simple:
1. The gradient points *uphill* (toward more loss). We want *less* loss, so we step in the **opposite** direction.
2. We take a small step — the step size is the **learning rate**.
3. Repeat thousands of times. The loss rolls downhill like a ball settling into a valley.

An **optimizer** (we'll use `Adam`) does the nudging for us. The cycle is always the same four moves:

> **forward** (predict) -> **loss** (measure) -> **backward** (find slopes) -> **step** (nudge dials)

*(Module 5.2 dissects optimizers in detail — momentum, AdamW, learning-rate pitfalls. Here we just run the loop.)*

## 7. Let's actually train one!

**The task:** teach our tiny network to copy the function $y = \sin(x)$ on the range $[-3, 3]$. The network has never seen a sine wave. It starts with random weights and only ever sees `(x, y)` pairs. Can gradient descent shape those random dials into a sine curve? Let's find out.

In [ ]:
# --- The data: 100 points along a sine wave ---
x_train = torch.linspace(-3, 3, 100).unsqueeze(1)  # shape (100, 1)
y_train = torch.sin(x_train)                        # the target we want to learn

# --- A fresh tiny network + an optimizer holding its parameters ---
torch.manual_seed(0)
net = nn.Sequential(
    nn.Linear(1, 32), nn.ReLU(),
    nn.Linear(32, 32), nn.ReLU(),
    nn.Linear(32, 1),
)
loss_fn   = nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)  # lr = step size

# --- THE TRAINING LOOP (the four sacred moves) ---
loss_history = []
for step in range(1500):
    optimizer.zero_grad()            # 0. clear old gradients (they accumulate otherwise)
    preds = net(x_train)             # 1. FORWARD:  predict
    loss  = loss_fn(preds, y_train)  # 2. LOSS:     how wrong?
    loss.backward()                  # 3. BACKWARD: compute every gradient (autograd)
    optimizer.step()                 # 4. STEP:     nudge every weight downhill

    loss_history.append(loss.item())
    if step % 300 == 0 or step == 1499:
        print(f"Step {step:4d} | Loss: {loss.item():.5f}")

print("\nThe loss fell from ~0.5 (random guessing) toward ~0 (learned the curve).")

### Watch the loss fall
A falling loss curve is the heartbeat of every neural network you'll ever train — from this toy to a 400-billion-parameter LLM. Let's plot it, and overlay the network's learned curve on the true sine wave.

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Left: the loss curve falling over training
ax1.plot(loss_history)
ax1.set_title("Loss falling during training")
ax1.set_xlabel("training step")
ax1.set_ylabel("loss (lower = better)")

# Right: did it actually learn the sine wave?
with torch.no_grad():                      # no gradients needed just to look
    learned = net(x_train)
ax2.plot(x_train.squeeze(), y_train.squeeze(), label="true sin(x)", linewidth=3, alpha=0.5)
ax2.plot(x_train.squeeze(), learned.squeeze(), label="network's guess", linestyle="--")
ax2.set_title("What the network learned")
ax2.set_xlabel("x"); ax2.set_ylabel("y")
ax2.legend()

plt.tight_layout()
plt.show()

That dashed line tracing the sine wave is a network that started as pure random noise. Nobody told it the formula for sine. Gradient descent alone shaped its parameters until it matched. **This is all that learning is.**

## 8. Train vs. validation (and overfitting)

One catch: a network can "learn" by **memorizing** the exact examples it saw instead of grasping the underlying pattern. That's called **overfitting** — it aces the training data but flops on anything new.

To catch this, we hold out some data the network *never trains on* — the **validation set** — and check the loss on it separately. If training loss keeps dropping but validation loss starts *rising*, the network is memorizing rather than generalizing. Watching both is how you know whether your model has actually learned something useful. We'll return to this throughout the course.

## So... what's an LLM?

An **LLM is a very large, specialized neural network** — millions of neurons in stacked layers, billions of parameters — trained by this *exact same loop* you just ran:

> forward -> loss -> backward -> step, repeated trillions of times.

The only differences are scale and the task: instead of fitting a sine wave, an LLM is trained to **predict the next token** in text. The neuron, the parameter, the loss, the gradient descent — every concept here is exactly what powers GPT and Llama.

The rest of this course builds that machine, piece by piece. You now have the mental model to follow along.

### 🏋️ Try it yourself

1. **Change the target function.** Swap `torch.sin(x_train)` for something else — try `x_train ** 2` (a parabola) or `torch.cos(x_train * 2)`. Retrain and re-plot. Does the network still nail it?
2. **Starve the network.** Shrink the hidden layers from `32` neurons down to `2`. Retrain. With too few parameters (too few dials) it *can't* bend enough to match the curve — that's **underfitting**. Watch the loss get stuck.
3. **Mess with the learning rate.** Try `lr=1.0` (too big — the loss may bounce or explode) and `lr=0.0001` (too small — learning crawls). Find the sweet spot.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
torch.manual_seed(0)

# Starter: change the target, the network size, or the learning rate.
x_train = torch.linspace(-3, 3, 100).unsqueeze(1)
y_train = torch.sin(x_train)            # TODO 1: try x_train ** 2  or  torch.cos(x_train * 2)

net = nn.Sequential(
    nn.Linear(1, 32), nn.ReLU(),        # TODO 2: shrink 32 -> 2 to see underfitting
    nn.Linear(32, 32), nn.ReLU(),
    nn.Linear(32, 1),
)
loss_fn   = nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.01)  # TODO 3: try 1.0 or 0.0001

for step in range(1500):
    optimizer.zero_grad()
    loss = loss_fn(net(x_train), y_train)
    loss.backward()
    optimizer.step()

print(f"Final loss: {loss.item():.5f}")
with torch.no_grad():
    plt.plot(x_train.squeeze(), y_train.squeeze(), label="target", linewidth=3, alpha=0.5)
    plt.plot(x_train.squeeze(), net(x_train).squeeze(), "--", label="network")
plt.legend(); plt.show()